## HIL-005 Data Cleaning

This notebook documents the cleaning and validation process for dataset HIL-005.  The raw source data will remain unchanged, and all cleaned outputs will be saved separately.

Source: City of Hillsboro GIS

Dataset: Buildings (<1,000)

Layer ID: 91

Geometry: Polygon

Spatial Reference: WKID 3857

Source documentation: [\[gis.hillsboro-oregon.gov\]](https://gis.hillsboro-oregon.gov/public/rest/services/public/Planning_BaseData/MapServer/91)

## Findings from the HIL_005_Buildings_Exploration Notebook

- The layer contains 43,686 building records.
- Geometry type is polygon.
- The current dataset contains no Demoed or Permitted records.
- `YEAR_BUILT = 0` occurs in 8,732 records (~20% of the dataset) and should be treated as a potential missing/unknown value rather than a literal construction year.
- `YEAR_DEMOLISHED` is not populated in the current dataset.
- The `STATUS` field uses a coded domain: 0 = Active, 1 = Demoed, 2 = Permitted.
- The current dataset therefore appears most useful for analyzing the existing building stock rather than historical demolition activity.

In [ ]:
from pathlib import Path

# Establish the project root
PROJECT_ROOT = Path.cwd().parent

# Locate available dated data folders
DATA_FOLDERS = sorted(
    [
        folder
        for folder in PROJECT_ROOT.iterdir()
        if folder.is_dir() and folder.name.startswith("20")
    ]
)

print("Available data folders:")
for folder in DATA_FOLDERS:
    print("-", folder.name)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Change to the desired data version here
DATA_VERSION = "2026-08-26"

DATA_ROOT = PROJECT_ROOT / DATA_VERSION
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"

print("Using data version:", DATA_VERSION)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

In [ ]:
# List files in the raw data directory
for file in RAW_DIR.rglob("*"):
    if file.is_file():
        print(file.relative_to(RAW_DIR))

In [ ]:
import json

# Define the raw HIL-005 file
RAW_FILE = RAW_DIR / "HIL-005.json"

# Load the raw data
with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("Loaded:", RAW_FILE.name)
print("Top-level type:", type(raw_data).__name__)

In [ ]:
# Inspect top-level keys and their value types
for key, value in raw_data.items():
    print(f"{key}: {type(value).__name__}")

In [ ]:
import pandas as pd

# Extract feature attributes into a DataFrame
df = pd.DataFrame(
    [feature["attributes"] for feature in raw_data["features"]]
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

In [ ]:
# Consider null values and data types for each column
df.info()

In [ ]:
# Distinguishing between missing values and empty values
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing["missing_percent"] = (
    missing["missing_count"] / len(df) * 100
)

missing.sort_values("missing_percent", ascending=False)

In [ ]:
df["SOURCE"].value_counts(dropna=False)

## `SOURCE` Assessment

The `SOURCE` field is fully populated across all 43,686 records and contains 29 distinct values. The observed values correspond to the coded values documented by the City of Hillsboro GIS layer.

The source values appear to encode the provenance and approximate date of the imagery or data source used to establish building information. The distribution is highly uneven, with `3DiJUL1999` accounting for approximately 47% of records.

### Cleaning Decision

No cleaning is currently required for the `SOURCE` field. The original source values will be preserved as provided by the City of Hillsboro GIS layer.

If human-readable source descriptions or temporal analysis are needed later, those should be added as derived fields rather than replacing the original values.

In [ ]:
df["YEAR_BUILT"].describe()

In [ ]:
# The mean is likely reduced from the presence of "0" values, and that the max is greater than the current year
print("YEAR_BUILT = 0:", (df["YEAR_BUILT"] == 0).sum())
print("YEAR_BUILT > 2026:", (df["YEAR_BUILT"] > 2026).sum())

In [ ]:
# What does the building with a YEAR_BUILT greater than 2026 look like? Are there any other anomalies in the data?
df.loc[df["YEAR_BUILT"] > 2026]

## `YEAR_BUILT` Anomaly Assessment

The `YEAR_BUILT` field contains 8,732 records with a value of `0` and one record with a value of `2029`.

The 2029 record has:
- `STATUS = 0` (Active)
- `SOURCE = Site Plan`
- `PERMIT_ID = CMB25-00176`

The combination of `SOURCE = Site Plan` and the presence of a building permit identifier provides context for the future year, but does not establish why `YEAR_BUILT` is recorded as 2029.

### Cleaning Decision

The 2029 value will be preserved. It will be treated as an anomalous value for quality-control purposes rather than automatically classified as erroneous or replaced with a null value.

In [ ]:
# What are the sources of the buildings with a YEAR_BUILT of 0? Are there any other anomalies in the data?
# Start by looking at the sources of the buildings with a YEAR_BUILT of 0
df.loc[df["YEAR_BUILT"] == 0, "SOURCE"].value_counts()

In [ ]:
# Calculate the percentage of buildings with YEAR_BUILT = 0 for each source
zero_by_source = df["SOURCE"].where(df["YEAR_BUILT"] == 0).value_counts()
total_by_source = df["SOURCE"].value_counts()

zero_percent_by_source = (
    zero_by_source / total_by_source * 100
).sort_values(ascending=False)

zero_percent_by_source

## `YEAR_BUILT` Assessment

The `YEAR_BUILT` field contains 8,732 records with a value of `0`, representing approximately 20% of the dataset. The value occurs across 25 of the 29 observed `SOURCE` values, but its prevalence varies substantially by source.

Some sources contain `YEAR_BUILT = 0` for nearly all records, while others contain very few or no zero values. This indicates that the use or availability of construction-year information varies by source.

### Cleaning Decision

No values in `YEAR_BUILT` will be modified at this stage. The original values, including `0` and `2029`, will be preserved while the meaning of the zero-value convention is investigated further.

## Cleaning: Convert Date Fields

The source schema identifies `UTC_CreateDate` and `UTC_EditDate` as date
fields. The downloaded JSON represents these values as Unix epoch
timestamps in milliseconds.

These fields will be converted to pandas datetime values while preserving
their UTC interpretation.

In [ ]:
# Convert the UTC_CreateDate and UTC_EditDate columns to datetime objects
date_columns = [
    "UTC_CreateDate",
    "UTC_EditDate"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        unit="ms",
        utc=True
    )

# Inspect the date columns after conversion
df[date_columns].info()
df[date_columns].head()

In [ ]:
# Inspect the min and max values of the date columns to check for anomalies
df[date_columns].agg(["min", "max"])

## Cleaning: YEAR_BUILT

Both CreateDate and EditDates look reasonable after transformation, and show no need for further manipulation.

However, YEAR_BUILT has shown a few anomalies and will likely need more in-depth cleaning.

`YEAR_BUILT` contains 8,732 records with a value of `0`. Because `0` is
not a valid construction year, these values are treated as unknown rather
than literal years.

One record contains `YEAR_BUILT = 2029`, which is beyond the current
calendar year but will remain in the dataset.

### Cleaning Rule

- Replace `YEAR_BUILT = 0` with a missing value.
- Retain `YEAR_BUILT = 2029` for now.
- Do not discard building records based solely on an unusual construction
  year.

In [ ]:
# Convert YEAR_BUILT values of 0 to NaN to avoid skewing the mean and other statistics
df["YEAR_BUILT"] = df["YEAR_BUILT"].replace(0, pd.NA)

print("YEAR_BUILT missing:", df["YEAR_BUILT"].isna().sum())

In [ ]:
# With the YEAR_BUILT values of 0 replaced with NaN, we can now check the data type of the 
# YEAR_BUILT column to ensure it is appropriate for further analysis.
df["YEAR_BUILT"].dtype

In [ ]:
# Convert YEAR_BUILT to a numeric type, coercing errors to NaN, and then convert to Int64 to allow for missing values
df["YEAR_BUILT"] = pd.to_numeric(
    df["YEAR_BUILT"],
    errors="coerce"
).astype("Int64")

In [ ]:
df["YEAR_BUILT"].dtype

In [ ]:
# Lastly, check the number of missing values in the YEAR_BUILT column after the conversion to Int64
print("YEAR_BUILT missing:", df["YEAR_BUILT"].isna().sum())

In [ ]:
# Check the data type, number of missing values, and min/max values of the YEAR_BUILT column
print("Dtype:", df["YEAR_BUILT"].dtype)
print("Missing:", df["YEAR_BUILT"].isna().sum())
print("Minimum:", df["YEAR_BUILT"].min())
print("Maximum:", df["YEAR_BUILT"].max())

### Validation Note

After cleaning, `YEAR_BUILT` contains 9,827 missing values and ranges from
1001 to 2029. The minimum value of 1001 is anomalous for this dataset and
should be investigated separately rather than automatically removed.

## Cleaning: STATUS

The source defines `STATUS` as a coded domain:

- `0` = Active
- `1` = Demoed
- `2` = Permitted

The raw coded value will be preserved. A new `STATUS_LABEL` field will be
created to provide a human-readable representation.

In [ ]:
status_labels = {
    "0": "Active",
    "1": "Demoed",
    "2": "Permitted"
}

df["STATUS_LABEL"] = df["STATUS"].map(status_labels)

In [ ]:
df[["STATUS", "STATUS_LABEL"]].value_counts()

In [ ]:
# Verify that the new mapping of STATUS to STATUS_LABEL is correct by checking the number of missing values in both columns
print("STATUS missing:", df["STATUS"].isna().sum())
print("STATUS_LABEL missing:", df["STATUS_LABEL"].isna().sum())

In [ ]:
df[["STATUS", "STATUS_LABEL"]].drop_duplicates()

## Data Quality Review: Field Cardinality

Before applying additional cleaning rules, examine the number of unique
values in each field. Fields with very few unique values may represent
coded domains or categorical information, while fields with little or no
variation may provide limited analytical value.

In [ ]:
df.nunique(dropna=False).sort_values()

## Field Classification

Each field is classified according to its role in the source dataset.
Classification will guide subsequent cleaning decisions while preserving
source information unless there is a documented reason to transform or
remove it.

Categories include:

- Identifier
- Categorical
- Numeric
- Date
- Provenance / Metadata
- Geometry-derived
- Completely Missing

In [ ]:
field_classification = {
    "OBJECTID": "Identifier",
    "BLDG_ID": "Identifier",
    "GlobalID": "Identifier",
    
    "STATUS": "Categorical",
    "STATUS_LABEL": "Categorical",
    "SOURCE": "Provenance / Metadata",
    "ROOF_TYPE": "Categorical",
    "DEMO_PERMIT": "Identifier",
    "PLANREFID": "Identifier",
    "PERMIT_ID": "Identifier",
    "OMS_FACILITY_ID": "Identifier",
    
    "NUM_STORIES": "Numeric",
    "HEIGHT": "Numeric",
    "YEAR_BUILT": "Numeric",
    "YEAR_DEMOLISHED": "Numeric",
    
    "UTC_CreateDate": "Date",
    "UTC_EditDate": "Date",
    
    "Tracking_CreateID": "Provenance / Metadata",
    "Tracking_EditID": "Provenance / Metadata",
    
    "CONST_TYPE": "Completely Missing",
    "ROOF_COVER": "Completely Missing",
    "BASEMENT": "Completely Missing",
    "SPRINKLED": "Completely Missing",
    "NFIRSCD": "Completely Missing",
    "OCCUPANCY_TYPE": "Completely Missing",
    
    "Shape.STArea()": "Geometry-derived",
    "Shape.STLength()": "Geometry-derived",
}

In [ ]:
classification = pd.Series(field_classification, name="classification")

classification

In [ ]:
# Table describing the fields in the DataFrame, their classifications, data types, number of non-null values, and number of unique values
field_audit = pd.DataFrame({
    "field": df.columns,
    "classification": [
        field_classification.get(field, "Unclassified")
        for field in df.columns
    ],
    "dtype": [
        str(df[field].dtype)
        for field in df.columns
    ],
    "non_null": [
        df[field].notna().sum()
        for field in df.columns
    ],
    "unique": [
        df[field].nunique(dropna=False)
        for field in df.columns
    ]
})

field_audit

In [ ]:
# Verifies that all fields in the DataFrame have been classified, and identifies any unclassified fields
unclassified = [
    field for field in df.columns
    if field not in field_classification
]

unclassified

## Retention Decision: Completely Missing Fields

Several fields contain no populated values in the current dataset:

- `NUM_STORIES`
- `NFIRSCD`
- `OCCUPANCY_TYPE`
- `SPRINKLED`
- `BASEMENT`
- `ROOF_COVER`
- `CONST_TYPE`
- `YEAR_DEMOLISHED`

These fields will be retained in the source-faithful dataset for
provenance and documentation purposes.

They are candidates for exclusion from a future analytical dataset if they
remain completely unpopulated.

No source fields will be removed during this initial cleaning stage solely
because they are empty.

In [ ]:
# Checking for anomalies in the HEIGHT column, such as negative values or unusually high/low values
df["HEIGHT"].describe()

In [ ]:
# Attain the number of buildings with a HEIGHT of 0, as well as the number of missing values in the HEIGHT column
height_zero = (df["HEIGHT"] == 0).sum()

print("HEIGHT = 0:", height_zero)
print("HEIGHT missing:", df["HEIGHT"].isna().sum())

In [ ]:
# Check to see if Heights of 0 are associated with a particular attribute
df.loc[
    df["HEIGHT"] == 0,
    ["BLDG_ID", "STATUS", "YEAR_BUILT", "SOURCE"]
].head(20)

In [ ]:
# Count the number of buildings with a HEIGHT of 0 for each source
df.loc[df["HEIGHT"] == 0, "SOURCE"].value_counts()

In [ ]:
df.loc[df["HEIGHT"] > 80, ["BLDG_ID", "HEIGHT", "YEAR_BUILT", "SOURCE"]].sort_values(
    "HEIGHT",
    ascending=False
)

### Validation Findings

`HEIGHT` contains 10,996 records with a value of `0`. These records are
distributed across numerous source datasets rather than being isolated to a
single source.

Because a zero-foot building height is unlikely to represent a literal
building height, these values are considered potential missing/sentinel
values. The source documentation should be consulted before converting them
to missing values.

The maximum observed height is 95.91 feet. Only two records exceed 80 feet,
with heights of 95.91 and 90.00 feet. These values are not considered
anomalous based on the available evidence and will be retained.

## Investigation: Large Buildings

Two buildings were identified as having heights greater than 80 feet.
Their `BLDG_ID` values will be cross-referenced against the City's address
layer using `BuildingID`.

This is an exploratory lookup only and does not modify the HIL-005 dataset.

In [ ]:
import requests

In [ ]:
address_url = (
    "https://gis.hillsboro-oregon.gov/public/rest/services/public/"
    "PW_FiberAddresses/MapServer/0/query"
)

params = {
    "where": "BuildingID IN (94866, 100710)",
    "outFields": "BuildingID,FullAddress,SubPudComplexName,X_LONG,Y_LAT",
    "returnGeometry": "false",
    "f": "json"
}

response = requests.get(address_url, params=params)
response.raise_for_status()

address_data = response.json()

address_data

In [ ]:
for feature in address_data["features"]:
    print(feature["attributes"])

In [ ]:
df.loc[
    df["BLDG_ID"].isin([94866, 100710]),
    ["BLDG_ID", "HEIGHT", "YEAR_BUILT", "SOURCE", "Shape.STArea()", "Shape.STLength()"]
]

### Investigation Findings

Two records exceeded 80 feet in height:

- `BLDG_ID 94866`: 95.91 feet
- `BLDG_ID 100710`: 90.00 feet

Both records were cross-referenced against the City's address layer and
visually inspected using external mapping imagery. The associated locations
appear to be residential structures rather than buildings approaching
90–96 feet in height.

Their building footprints are also relatively modest:

- `BLDG_ID 94866`: 1,767.7 square feet
- `BLDG_ID 100710`: 1,210.5 square feet

These observations provide strong evidence that the recorded heights may be
erroneous. The records will be flagged for further investigation rather than
silently corrected, since the underlying cause of the anomalous values has
not yet been established.

In [ ]:
from pathlib import Path
import json

project_root = Path.cwd().parent
DATA_VERSION = "2026-08-26"

raw_path = project_root / DATA_VERSION / "raw" / "HIL-005.json"

with open(raw_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded: {raw_path}")
print(f"Features: {len(data['features']):,}")

In [ ]:
# Verifying the address of the buildings with BLDG_IDs 94866 and 100710 by checking the address data returned from the GIS service
target_ids = {94866, 100710}

target_features = [
    feature
    for feature in data["features"]
    if feature["attributes"]["BLDG_ID"] in target_ids
]

print(f"Found: {len(target_features)} features")

for feature in target_features:
    attrs = feature["attributes"]
    print(
        attrs["BLDG_ID"],
        attrs["HEIGHT"],
        attrs["Shape.STArea()"],
        feature["geometry"]["rings"][0][:2]
    )

In [ ]:
data["spatialReference"]

In [ ]:
from pyproj import Transformer

In [ ]:
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True
)

In [ ]:
address_points = {
    94866: (-122.99794163, 45.53854491),
    100710: (-122.93632363, 45.49744841),
}

projected_points = {}

for building_id, (lon, lat) in address_points.items():
    x, y = transformer.transform(lon, lat)
    projected_points[building_id] = (x, y)
    print(building_id, x, y)

In [ ]:
from shapely.geometry import Point, Polygon

In [ ]:
from shapely.geometry import Polygon

building_polygons = {}

for feature in target_features:
    attrs = feature["attributes"]
    building_id = attrs["BLDG_ID"]
    ring = feature["geometry"]["rings"][0]
    
    building_polygons[building_id] = Polygon(ring)

for building_id, polygon in building_polygons.items():
    print(
        f"BLDG_ID {building_id}: "
        f"valid={polygon.is_valid}, "
        f"area={polygon.area:.2f}"
    )

## Height Investigation — Polygon Validation

The two buildings identified as height anomalies (`BLDG_ID` 94866 and `BLDG_ID` 100710) were retrieved from the original HIL-005 geometry and converted to Shapely polygons.

Both reconstructed polygons are geometrically valid. However, the planar areas calculated by Shapely do not match the `Shape.STArea()` values provided by the ArcGIS dataset:

- `BLDG_ID 94866`: Shapely area = 334.77 vs. `Shape.STArea()` = 1,767.73
- `BLDG_ID 100710`: Shapely area = 228.90 vs. `Shape.STArea()` = 1,210.47

This discrepancy does not necessarily indicate an error in the geometry. ArcGIS and Shapely may calculate or interpret geometry measurements differently, particularly with projected GIS data.

For the current investigation, the area discrepancy does not prevent us from testing the proposed relationship between `BLDG_ID` and the address-layer `BuildingID`. The next step is to compare the projected address coordinates with the actual building polygon locations.

**Finding:** The polygon geometries are valid, but their calculated areas require additional interpretation before using area measurements for QA or analysis.

In [ ]:
for building_id, polygon in building_polygons.items():
    print(f"\nBLDG_ID {building_id}")
    print(f"Bounds: {polygon.bounds}")
    print(f"Area: {polygon.area:.2f}")

In [ ]:
from shapely.geometry import Point

for building_id, point_coords in projected_points.items():
    polygon = building_polygons[building_id]
    point = Point(point_coords)

    print(
        f"BLDG_ID {building_id}: "
        f"inside={polygon.contains(point)}, "
        f"distance={polygon.distance(point):.2f} meters"
    )

## Height Investigation — Building ID Relationship Validation

The relationship between the HIL-005 `BLDG_ID` field and the `BuildingID` field used in the address lookup was validated spatially for the two buildings identified as height anomalies.

The address coordinates were originally returned in WGS 84 (EPSG:4326) and were projected to the HIL-005 spatial reference (EPSG:3857 / WKID 102100). The resulting address points were then compared with the corresponding HIL-005 building polygons.

Results:

- `BLDG_ID 94866`: address point falls inside the HIL-005 building polygon.
- `BLDG_ID 100710`: address point falls inside the HIL-005 building polygon.

Both tests returned `inside=True` and a polygon distance of `0.00` meters.

**Finding:** The matching `BLDG_ID` / `BuildingID` relationship is supported by spatial evidence for both investigated records. Using the building ID to retrieve their addresses was therefore a reasonable method for investigating the anomalous height values.

This validation applies specifically to the records tested and should not automatically be interpreted as proof that the relationship is valid for every record in the two datasets.

In [ ]:
# Flag unusually tall buildings for further review
df["HEIGHT_ANOMALY"] = df["HEIGHT"] > 80

print(df["HEIGHT_ANOMALY"].value_counts())

In [ ]:
df.loc[
    df["HEIGHT_ANOMALY"],
    ["BLDG_ID", "HEIGHT", "YEAR_BUILT", "SOURCE"]
]

## Cleaning: HEIGHT

`HEIGHT` was reviewed for missing values, zero values, unusually large values, and potential anomalies.

- 4,461 records have a missing `HEIGHT`.
- 10,996 records contain `HEIGHT = 0`.
- Two records contain heights greater than 80 feet:
  - `BLDG_ID 94866`: 95.91 feet
  - `BLDG_ID 100710`: 90.00 feet
- Both records were investigated using the associated building ID and an independent address dataset.
- The `BLDG_ID` / `BuildingID` relationship was spatially validated: the retrieved address points fall within the corresponding HIL-005 building polygons.
- No authoritative source was identified that establishes replacement height values for these records.

**Decision:** Retain the original height values and flag the two records as height anomalies rather than altering or deleting potentially valid source data.

The `HEIGHT_ANOMALY` field is an analytical flag and does not replace the source `HEIGHT` value.

## Cleaning: SOURCE — Initial Review

The `SOURCE` field identifies the apparent source or origin of each building record. Unlike fields such as `HEIGHT` or `YEAR_BUILT`, `SOURCE` is not simply a value to validate numerically; it is a provenance field and should be preserved carefully.

The initial review will focus on:

1. **Completeness** — determine whether any records lack a `SOURCE` value.
2. **Consistency** — identify distinct values and check for unexpected variations in capitalization, spacing, or formatting.
3. **Frequency** — determine how many building records are associated with each source.
4. **Meaning** — investigate what the source codes represent, including embedded dates or abbreviations.
5. **Temporal context** — compare source dates with `YEAR_BUILT` and other relevant fields where appropriate.
6. **Standardization** — determine whether the source values can be represented in a more useful, consistent form without losing the original provenance information.

The original `SOURCE` values will be retained. Any standardized or derived provenance fields will be created separately so that the source data remains traceable.

The goal is not to make every source value look uniform simply for the sake of consistency. Instead, the goal is to make the provenance information easier to interpret while preserving what the original dataset tells us.

In [ ]:
# Check how many building records are missing a SOURCE value.
# This tells us whether the provenance field is complete.
print(f"SOURCE missing: {df['SOURCE'].isna().sum():,}")

In [ ]:
# Display the distinct SOURCE values in alphabetical order.
# This makes it easier to identify naming patterns such as prefixes, dates, and
# inconsistent formatting before we decide whether any standardization is needed.
print(sorted(df["SOURCE"].unique()))

In [ ]:
# Extract a four-digit year from each SOURCE value where a year is present.
# This lets us separate the source's apparent date from the original SOURCE label
# without modifying the original provenance field.
df["SOURCE_YEAR"] = (
    df["SOURCE"]
    .str.extract(r"(\d{4})", expand=False)
    .astype("Int64")
)

# Display the distinct SOURCE values alongside their extracted years.
# This helps us verify that the extraction works across the different naming patterns.
print(
    df[["SOURCE", "SOURCE_YEAR"]]
    .drop_duplicates()
    .sort_values(["SOURCE_YEAR", "SOURCE"])
    .to_string(index=False)
)

In [ ]:
# Identify records where the apparent SOURCE year is later than YEAR_BUILT.
# A source created after construction is normal, so this is not an anomaly by itself.
# Instead, we are looking for the opposite relationship: a source dated before the
# recorded construction year, which could indicate a questionable YEAR_BUILT value.
source_before_build = df[
    df["SOURCE_YEAR"].notna()
    & df["YEAR_BUILT"].notna()
    & (df["SOURCE_YEAR"] < df["YEAR_BUILT"])
]

print(f"Records where SOURCE_YEAR < YEAR_BUILT: {len(source_before_build):,}")

In [ ]:
# Display the records where the apparent source date precedes the recorded
# construction year so we can determine whether the relationship is plausible
# or indicates another data-quality issue.
source_before_build[
    ["BLDG_ID", "YEAR_BUILT", "SOURCE", "SOURCE_YEAR"]
].head(20)

In [ ]:
# Count how many records have a SOURCE_YEAR earlier than YEAR_BUILT.
# This tells us whether the apparent temporal mismatch is an isolated issue
# or a common characteristic of the building dataset.
print(f"Records where SOURCE_YEAR < YEAR_BUILT: {len(source_before_build):,}")
print(
    f"Percentage of dataset: "
    f"{len(source_before_build) / len(df) * 100:.2f}%"
)

# Count the source values among records where SOURCE_YEAR precedes YEAR_BUILT.
# This helps determine whether the pattern is concentrated in particular source datasets.
print(source_before_build["SOURCE"].value_counts())

## SOURCE — Temporal Cross-Check

The apparent year embedded in `SOURCE` was extracted into a derived `SOURCE_YEAR` field and compared with `YEAR_BUILT`.

There are 421 records (0.96% of the dataset) where `SOURCE_YEAR` is earlier than `YEAR_BUILT`.

These records are concentrated primarily among several older and newer source groups, including `3DiJUL1999` and multiple `GEOTERRA` sources.

This relationship does not necessarily indicate an error. A source year may describe the date of an imagery or source dataset rather than the date when the building record was created or when its `YEAR_BUILT` value was established. Therefore, a source predating the recorded construction year can be legitimate depending on how the source field was used in the City's data-maintenance process.

**Decision:** Do not modify `YEAR_BUILT` or `SOURCE` based solely on this comparison. `SOURCE_YEAR` will be treated as contextual provenance information rather than a validation constraint against `YEAR_BUILT`.

In [ ]:
# Extract the source-family prefix by removing the optional month abbreviation
# and four-digit year from the end of each SOURCE value.
# The original SOURCE field is preserved unchanged for provenance.
df["SOURCE_PREFIX"] = (
    df["SOURCE"]
    .str.replace(r"(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)?\d{4}$", "", regex=True)
    .str.strip()
)

# Display each original SOURCE value alongside its extracted source family.
# This lets us verify that the parsing worked correctly across all naming patterns.
print(
    df[["SOURCE", "SOURCE_PREFIX"]]
    .drop_duplicates()
    .sort_values(["SOURCE_PREFIX", "SOURCE"])
    .to_string(index=False)
)

In [ ]:
# Count building records by the extracted source family.
# This shows the overall contribution of each source family to the dataset.
source_prefix_counts = df["SOURCE_PREFIX"].value_counts()

print(source_prefix_counts)

## SOURCE — Prefix Classification

The `SOURCE` field contains 29 distinct values. These values follow a consistent pattern in which many combine a source-family prefix with an apparent month and/or year.

A derived `SOURCE_PREFIX` field was created by removing the optional month abbreviation and four-digit year from the original `SOURCE` value. The original `SOURCE` field was retained unchanged.

The 29 source values resolve into 10 source families:

- `3Di` — 20,445 records
- `iTEN` — 5,786 records
- `PIX` — 5,034 records
- `GEOTERRA` — 4,678 records
- `DOGAMI` — 2,883 records
- `OSI` — 2,867 records
- `SAN` — 1,626 records
- `Site Plan` — 253 records
- `LiDAR` — 109 records
- `SBG` — 5 records

The extracted source families account for all 43,686 building records.

### Source Interpretation

Targeted research was conducted to determine what the source-family prefixes represent. Some interpretations can be supported by authoritative or dataset-specific evidence, while others could not be verified.

**Higher-confidence interpretations:**

- `GEOTERRA` — associated with GeoTerra's aerial mapping, GIS, and surveying work.
- `DOGAMI` — Oregon Department of Geology and Mineral Industries.
- `LiDAR` — identifies a LiDAR-derived source or data type.
- `Site Plan` — identifies a site-plan-derived source.

**Unverified interpretations:**

- `3Di`
- `iTEN`
- `PIX`
- `OSI`
- `SAN`
- `SBG`

Although organizations or technologies with names matching some of these prefixes can be found through external research, no sufficiently strong Hillsboro-specific evidence was identified to establish those interpretations as fact.

**Decision:** `SOURCE` will be preserved exactly as provided by the original dataset. `SOURCE_PREFIX` provides a standardized grouping mechanism, while `SOURCE_YEAR` provides the apparent year encoded in the source label. Organization-level interpretations will not be written into the cleaned dataset unless they can be independently verified.

This approach preserves the original provenance information while avoiding unsupported assumptions about legacy source codes.